# Initalize libraries

## Import libraries

In [ ]:
import sys, os
import time
from os.path import join
from os import path
from importlib import reload
from getpass import getuser
from glob import glob

import xarray as xr
import h5py
from tqdm.auto import tqdm

import numpy as np
import matplotlib.pyplot as plt
import scipy
import fabio
import skimage.morphology

# Open nexus files
from nexusformat.nexus import *

from scipy.ndimage.filters import gaussian_filter

# pyFAI
import pyFAI
from pyFAI.azimuthalIntegrator import AzimuthalIntegrator
from pyFAI.detectors import Detector

# Self-written libraries
sys.path.append(join(os.getcwd(), "library"))
import mask_lib
import reconstruct as reco
import fthcore as fth
import helper_functions as helper
from interactive import cimshow
import interactive

# Gifs
import imageio

from scipy import stats

plt.rcParams["figure.constrained_layout.use"] = True  # replaces plt.tight_layout

import reconstruct as reco
import reconstruct_rb as rec
import phase_retrieval_core as PhR

In [ ]:
# Is there a GPU?
try:
    # Cupy
    import cupy as cp
    import cupyx as cpx

    GPU = True

    print("GPU available")

    # Self-written library
    import CCI_core_cupy as cci
except:
    GPU = False
    import CCI_core as cci

    print("GPU unavailable")

In [ ]:
# interactive plotting
import ipywidgets

%matplotlib widget

# Auto formatting of cells
#%load_ext jupyter_black

In [ ]:
facility = "MAXIV" # Options: "SwissFEL", "MAXI"
BEAMTIMEID = 2026021708 # Proposal number
data_fname_prefix = "2602_softimax"
USER = getuser()

# Facility specific loading functions
if facility == "PETRA":
    import PETRA_MaxP04_loading as loading
elif facility == "MAXI":
    import MAXI_loading as loading
elif facility == "SwissFEL":
    from sfdata import SFDataFiles, SFScanInfo, SFProcFile
    import Swiss_FEL_Loading as loading

    # Number or jobs for analysis
    NR_JOBS = 32
elif facility == "MAXIV":
    import MAXI_loading as loading

BASEFOLDER = "/data/visitors/softimax/20250671/%d"%BEAMTIMEID
DATAFOLDER = join(BASEFOLDER,"raw")

print("Raw Datafolder is: %s"%DATAFOLDER)

# Load dictionary for keys etc
mnemonics = loading.load_mnemonics()

### Loading data

In [ ]:
def generate_filename(scan_nr: int):
    """
    Generates filename of the given scan id

    Parameter
    =========
    scan_nr : number identifier (id) of the given scan

    Output
    ======
    filename : str
        full generated filename
    ======
    author: ck 2025
    """
    return join(DATAFOLDER, f"{data_fname_prefix}_{scan_nr:04d}.h5")

def load_key(scan_id, key):
    """
    Load any kind of data specified by key (path)
    
    Parameter
    =========
    scan_id : int
        experimental identifier of scan
    key : str
        key path of nexus file tree to relevant data field
   
    Output
    ======
    data : dict
        data dictionaray on single key
    ======
    author: ck 2024
    """
    #Generate filename from scan_id
    fname = generate_filename(scan_id)
    
    # load data with basic loading function
    data = loading.load_key(fname, key)
    
    return data

def load_data(scan_id, keypath=mnemonics["measurement"], keys=None):
    """
    Load data of all specified keys from keypath

    Parameter
    =========
    scan_id : int
        experimental identifier of scan
    keypath : str
        path of nexus file tree to relevant data field
    keys : str or list of strings
        keys to load from keypath

    Output
    ======
    data : dict
        data dictionary of keys
    ======
    author: ck 2024
    """

    # Generate filename from scan_id
    fname = generate_filename(scan_id)

    # load data with basic loading function
    data = loading.load_data(fname, keypath, keys=keys)

    return data


## Loading images

In [ ]:
def get_filepath(tiffname, camera_type = "cmos"):
    """Return raw data path from save file path."""
    head, tifffile = path.split(tiffname)
    head, scandir = path.split(head)
    return path.join(DATAFOLDER, camera_type, scandir, tifffile)

def load_tiff(tifffile):
    """load single tiff image into np array"""
    return np.array(Image.open(tifffile))

def load_tiff_list(filelist, processes=None):
    """Multiprocessing loading of image files"""
    with Pool(processes=processes) as pool:
        frames = pool.map(load_tiff, filelist)
    return np.stack(frames)

def load_cmos(scanid, file_indices = None):
    # Loads all single frames
    fname = generate_filename(scanid)
    cmos_filelist = load_key(scanid,mnemonics["cmos"])
    cmos_filelist = [get_filepath(b.decode(),camera_type = "cmos") for b in cmos_filelist]

    if file_indices is not None:
        cmos_filelist = [cmos_filelist[index] for index in file_indices]

    #print("Loading %d single frames"%len(cmos_filelist))
    return load_tiff_list(cmos_filelist)

In [ ]:
def load_images(im_id: int, file_indices=None, camera_type: str = "ccd"):
    """
    Load images corresponding to a given experimental image ID.

    Parameters
    ----------
    im_id : int
        Experimental identifier of the image scan.
    file_indices : array-like or None, optional
        Indices of frames to select from the image stack.
        If None, all frames are returned.
    camera_type : {"ccd", "cmos"}, optional
        Camera type used for acquisition.

    Returns
    -------
    images : np.ndarray
        Image stack with shape (n_frames, height, width).

    Raises
    ------
    ValueError
        If an unsupported camera_type is specified.
    ======
    author: ck 2024
    """

    if camera_type == "ccd":
        meta = load_data(im_id)

        try:
            raw_name = meta["ccd"][0]
        except (KeyError, IndexError) as exc:
            raise ValueError(f"No CCD data found for im_id={im_id}") from exc

        spe_path = (
            f"{BASEFOLDER}/raw/ccd/"
            f"{str(raw_name).split('//')[1].split('.spe')[0]}-raw.spe"
        )

        frames = imageio.mimread(spe_path, memtest="5000MB")
        images = np.squeeze(np.asarray(frames))

        if file_indices is not None:
            images = images[file_indices]

        zeros = images[...,5:]<3
        if np.sum(zeros) > 1000:
            print(f"Many zeros in image frames. Verify data transfer!")
            #raise ValueError(f"Many zeros in image frames. Verify data transfer!")
        
        images = np.stack(images)

    elif camera_type == "cmos":
        images = load_cmos(im_id, file_indices)

    else:
        raise ValueError(
            f"Unsupported camera_type '{camera_type}'. "
            "Valid options are 'ccd' and 'cmos'."
        )

    return images


### Loading image procedure

In [ ]:
# Full image loading procedure
def load_processing(im_id, file_indices = None, binning = 1, crop = None):
    """
    Loads images, averaging of two individual images (scans in tango consist of two images),
    padding to square shape, Additional cropping (optional)
    """

    # Load data
    images = load_images(im_id, file_indices = file_indices)

    # Optional cropping
    if crop is not None:
        images = images[..., :crop, :crop]

    # Binning
    if binning > 1:
        images = helper.binning(images, binning)

    # Average over all images
    if images.ndim == 4:
        image = np.mean(images, axis=(0, 1))
    elif images.ndim == 3:
        image = np.mean(images, axis=(0))
    elif images.ndim == 2:
        image = images.copy()
    images = np.stack(images)
    
    return image, images

### Loading, saving fth & cdi data

In [ ]:
# Saving of log files for fth and cdi recos
def save_fth_h5():
    # Save h5
    data = {}
    data["im_id"] = im_id
    data["topo_id"] = topo_id
    data["topo_centered"] = topo_c
    data["im_centered"] = im_c
    data["holo"] = holo
    data["recon"] = recon
    data["factor"] = factor
    data["offset"] = offset
    data["center"] = center
    data["roi"] = roi
    data["prop_dist"] = prop_dist
    data["phase"] = phase
    data["mask_bs"] = mask_pixel_smooth
    data["bs_smoothing"] = bs_smoothing
    data["experimental_setup"] = experimental_setup

    filename = join(
        folder_general, "Logs", "Data_ImId_%s_RefId_%s_%s" % (im_id, topo_id, USER)
    )
    print("Now Saving: %s" % filename)
    cci.create_hdf5(data, filename)


def save_cdi_h5():
    # Save h5
    data = {}
    data["im_id"] = im_id
    data["topo_id"] = topo_id
    data["pos"] = pos
    data["neg"] = neg
    data["factor"] = factor
    data["offset"] = offset
    data["center"] = center
    data["roi"] = roi
    data["prop_dist"] = prop_dist_cdi
    data["phase"] = phase_cdi
    data["mask_bs"] = mask_bs_cdi
    data["supportmask"] = supportmask
    data["mask_pixel"] = mask_pixel
    data["p_pc"] = p_pc
    data["n_pc"] = n_pc
    data["experimental_setup"] = experimental_setup

    filename = join(
        folder_general,
        "Logs",
        "Data_ImId_%s_RefId_%s_cdi_%s" % (im_id, topo_id, USER),
    )
    print("Now Saving: %s" % filename)
    cci.create_hdf5(data, filename)
    return

## Worker which performs complete fth reconstruction process

In [ ]:
def worker(image, topo, Norm = True):
    # Centering
    shift_c = np.array(topo.shape) / 2 - center
    topo_c = cci.shift_image(topo, shift_c)
    im_c = cci.shift_image(image, shift_c)

    ## Image registration
    shift = cci.image_registration(
       im_c[roi_im_reg],
        topo_c[roi_im_reg],
     method="dipy",
    )
    print("Relative shift is: %s" % shift)

    # Correct relative drift
    if sum(abs(shift)) > 0.05:
        im_c = cci.shift_image(im_c, -shift)
    
    if Norm:
        # Get scaling factor and offset
        factor, offset = cci.dyn_factor(
            im_c * (1 - mask_pixel),
            topo_c * (1 - mask_pixel),
            method="correlation",
            verbose=False,
            plot=False,
        )
    else:
        factor = 1
        offset = 0

    # Calculate differences (magnetic) and sums (topographc) contrast holograms.
    # _c: centered, without beamstop, _b: centered, with beamstop
    diff_c = im_c / factor - topo_c - offset
    sum_c = im_c / factor + topo_c - offset

    # Reconstruct
    recon = cci.reconstruct(
        cci.propagate(diff_c, prop_dist * 1e-6, experimental_setup=experimental_setup)
        * np.exp(1j * phase)
    )

    # worker dictionary
    worker_dict = {}
    worker_dict["center"] = center
    worker_dict["topo_c"] = topo_c
    worker_dict["im_c"] = im_c
    worker_dict["recon"] = recon
    worker_dict["factor"] = factor
    worker_dict["offset"] = offset
    worker_dict["shift"] = shift
    worker_dict["diff_c"] = diff_c
    worker_dict["sum_c"] = sum_c
    worker_dict["mask_pixel_smooth"] = mask_pixel_smooth
    worker_dict["mask_pixel"] = mask_pixel

    return worker_dict

In [ ]:
# Setup phase and propagation for cdi once
phase_cdi = 0
prop_dist_cdi = 0
dx = 0
dy = 0

def phase_retrieval_old(
    pos, neg, mask_pixel, supportmask, vmin=0, Startimage=None, Startgamma=None
):
    # Prepare Input holograms
    pos2 = pos.copy()
    neg2 = neg.copy()

    mi, _ = np.percentile(pos2[pos2 != 0], [vmin, 99.9])
    pos2 = pos2 - mi
    mi, _ = np.percentile(neg2[neg2 != 0], [vmin, 99.9])
    neg2 = neg2 - mi

    pos2[pos2 < 0] = 0
    neg2[neg2 < 0] = 0
    pos2 = pos2.astype(complex)
    neg2 = neg2.astype(complex)

    bsmask_p = mask_pixel.copy()
    bsmask_p[pos2 <= 0] = 1
    bsmask_n = mask_pixel.copy()
    bsmask_n[neg2 <= 0] = 1

    # Setup start image and startgamma
    if Startimage is None:
        Startimage = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(supportmask)))
    else:
        Startimage = Startimage.copy()
    if Startgamma is None:
        Startgamma = np.ones(pos.shape) * 1e-6 * 2
        Startgamma[pos.shape[0] // 2, pos.shape[1] // 2] = 0.7
    else:
        Startgamma = Startgamma.copy()

    # Settings for phase retrieval reconstructions
    partial_coherence = True

    # Setup
    retrieved_p = np.zeros(pos2.shape, np.cdouble)
    retrieved_n = np.zeros(pos2.shape, np.cdouble)

    # Algorithms and Inital guess
    plt.rcParams["figure.dpi"] = 100
    print("CDI - larger mask")

    algorithm_list = ["mine", "mine", "mine"]
    Nit_list = [700, 50, 50]  # iterations for algorithm_list

    x = (np.sqrt(np.maximum(pos2, np.zeros(pos2.shape)))[mask_pixel == 0]).flatten()
    y = ((np.abs(Startimage))[mask_pixel == 0]).flatten()
    res = stats.linregress(x, y)
    Startimage -= res.intercept
    Startimage /= res.slope

    average_img = 30
    real_object = False  # always set to False

    if partial_coherence:
        RL_freq = 20
        RL_it = 50

        algorithm_list_pc = ["mine", "ER", "ER"]
        Nit_list_pc = [700, 50, 50]

    # Execute Phase retrieval
    start_time = time.time()
    for i in range(len(Nit_list) // 3):
        print("############ -   CDI")

        # Positive helicity - beta_mode="arctan"
        retrieved_p, Error_diff_p, Error_supp = phr_gold.PhaseRtrv_GPU(
            diffract=np.sqrt(np.maximum(pos2, np.zeros(pos2.shape))),
            mask=supportmask,
            mode=algorithm_list[3 * i],
            beta_zero=0.5,
            Nit=Nit_list[3 * i],
            beta_mode="arctan",
            plot_every=349,
            Phase=Startimage,
            seed=False,
            real_object=real_object,
            bsmask=bsmask_p,
            average_img=average_img,
            Fourier_last=True,
        )

        # Positive helicity - beta_mode="const"
        retrieved_p, Error_diff_p2, Error_supp = phr_gold.PhaseRtrv_GPU(
            diffract=np.sqrt(np.maximum(pos2, np.zeros(pos2.shape))),
            mask=supportmask,
            mode=algorithm_list[3 * i + 1],
            beta_zero=0.5,
            Nit=Nit_list[3 * i + 1],
            beta_mode="const",
            plot_every=24,
            Phase=retrieved_p,
            seed=False,
            real_object=real_object,
            bsmask=bsmask_p,
            average_img=average_img,
            Fourier_last=True,
        )

        # Negative helicity - beta_mode="arctan"
        retrieved_n, Error_diff_n2, Error_supp = phr_gold.PhaseRtrv_GPU(
            diffract=np.sqrt(np.maximum(neg2, np.zeros(neg2.shape))),
            mask=supportmask,
            mode=algorithm_list[3 * i + 2],
            beta_zero=0.5,
            Nit=Nit_list[3 * i + 2],
            beta_mode="const",
            plot_every=24,
            Phase=retrieved_p * np.sqrt(np.sum(neg2) / np.sum(pos2)),
            seed=False,
            real_object=real_object,
            bsmask=bsmask_n,
            average_img=average_img,
            Fourier_last=True,
        )

        print("--- %s seconds ---" % np.round((time.time() - start_time), 2))

        Startimage = retrieved_p.copy()

        # Partial coherence phase retrieval
        if partial_coherence:
            # CDI_PC
            print("############   -   CDI_pc")
            pos3 = (np.abs(retrieved_p) ** 2) * bsmask_p + np.maximum(
                pos2, np.zeros(pos2.shape)
            ) * (1 - bsmask_p)
            neg3 = (np.abs(retrieved_n) ** 2) * bsmask_n + np.maximum(
                neg2, np.zeros(neg2.shape)
            ) * (1 - bsmask_n)

            # retrieve pos image
            (
                retrieved_p_pc,
                Error_diff_p_pc,
                Error_supp,
                gamma_p,
            ) = phr_gold.PhaseRtrv_with_RL(
                diffract=np.sqrt(pos3),
                mask=supportmask,
                mode=algorithm_list_pc[3 * i],
                beta_zero=0.5,
                Nit=Nit_list_pc[3 * i],
                beta_mode="arctan",
                gamma=Startgamma,
                RL_freq=RL_freq,
                RL_it=RL_it,
                plot_every=349,
                Phase=Startimage,
                seed=False,
                real_object=False,
                bsmask=np.zeros(bsmask_p.shape),
                average_img=average_img,
                Fourier_last=True,
            )

            (
                retrieved_p_pc,
                Error_diff_p_pc2,
                Error_supp,
                gamma_p,
            ) = phr_gold.PhaseRtrv_with_RL(
                diffract=np.sqrt(pos3),
                mask=supportmask,
                mode=algorithm_list[3 * i + 1],
                beta_zero=0.5,
                Nit=Nit_list_pc[3 * i + 1],
                beta_mode="const",
                gamma=gamma_p,
                RL_freq=RL_freq,
                RL_it=RL_it,
                plot_every=24,
                Phase=retrieved_p_pc,
                real_object=False,
                bsmask=np.zeros(bsmask_p.shape),
                average_img=average_img,
                Fourier_last=True,
            )
            (
                retrieved_n_pc,
                Error_diff_n_pc2,
                Error_supp,
                gamma_n,
            ) = phr_gold.PhaseRtrv_with_RL(
                diffract=np.sqrt(neg3),
                mask=supportmask,
                mode=algorithm_list[3 * i + 2],
                beta_zero=0.5,
                Nit=Nit_list_pc[3 * i + 2],
                beta_mode="const",
                gamma=gamma_p,
                RL_freq=RL_freq,
                RL_it=RL_it,
                plot_every=24,
                Phase=retrieved_p_pc * np.sqrt(np.sum(neg2) / np.sum(pos2)),
                real_object=False,
                bsmask=np.zeros(bsmask_n.shape),
                average_img=average_img,
                Fourier_last=True,
            )

            print("--- %s seconds ---" % np.round((time.time() - start_time), 2))

            Startimage = retrieved_p_pc.copy()
            Startgamma = gamma_p.copy()

    print("Phase Retrieval Done!")

    return (
        retrieved_p,
        retrieved_n,
        retrieved_p_pc,
        retrieved_n_pc,
        bsmask_p,
        bsmask_n,
        gamma_p,
        gamma_n,
    )

## Other

In [ ]:
def save_gif(output_path, image_path_list, fps=3 ):
    writer = imageio.get_writer(output_path, format="GIF-PIL", fps=fps)
    for im in tqdm(image_path_list):
        writer.append_data(imageio.imread(im))
    writer.close()

# Experimental Details

In [ ]:
# Dict with most basic experimental parameter
experimental_setup = {
    "ccd_dist": 0.09,  # ccd to sample distance
    "px_size": 20e-6,  # CMOS: 11 um, Sophia CCD: 13.5 um, Other CCD: 20µm
    "binning": 1,  # Camera binning
    "oversaturation": 2**16,  # Pixel saturation threshold
}

# Setup for azimuthal integrator
detector = Detector(
    experimental_setup["binning"] * experimental_setup["px_size"],
    experimental_setup["binning"] * experimental_setup["px_size"],
)

# General saving folder and log folder
folder_general = join(BASEFOLDER, "process")
helper.create_folder(folder_general)

print("Output Folder: %s" % folder_general)

# Load images

Start by loading the images: image of interest (im), reference of charge scattering (topo), any kind of dark image (dark)

We estalished the following convention: Difference Hologram which contains only the magnetic scattering will be calculated according to:

$Diff = \frac{Image}{factor} - Topo$,

where the factor is used for intensity scaling. In Case that you recorded scans of the same magnetic state with both helicities, use the image with negative helicity as topo and the one with positive helicity as image

In [ ]:
# Define scan ids for each image
im_id_set = [2936]  # single helicity mode: image with magnetic contrast, double helicity: pos
topo_id_set = (
    [2934]  # single helicity mode: image without magnetic contrast, double helicity: neg
)
dark_id_im_set = [2932]
dark_id_topo_set = dark_id_im_set

# Optional for CMOS data
file_indices_im = None #np.arange(50) # Load specific image stack, None means all stacks
file_indices_topo = None # Load specific topo stack, None means als

# Which other meta data to load
scan_axis = "magnetIP"

# Load energy and add to experimental setup
experimental_setup["energy"] = load_key(im_id_set[0], mnemonics["energy"])
experimental_setup["lambda"] = helper.photon_energy_wavelength(
    experimental_setup["energy"], input_unit="eV"
)

print("Image Id: %s" % im_id_set)
print("Topo Id: %s" % topo_id_set)
print("Dark Id: %s" % dark_id_im_set)



## Load image of interest

In [ ]:
# Load image
images = []
for i, im_id in enumerate(im_id_set):
    image, _ = load_processing(im_id, file_indices = file_indices_im, crop=None)
    dark, _ = load_processing(dark_id_im_set[i], file_indices = None, crop=None)
    image = image - dark
    images.append(image)
    
images = np.stack(images)
image = np.mean(images,axis=0)
    
# Plot
fig, ax = cimshow(images)
ax.set_title("Images")

## Load topo data set and average

In [ ]:
# Load topo
topos = []
for i, topo_id in enumerate(topo_id_set):
    topo, _ = load_processing(topo_id, file_indices = file_indices_topo, crop=None)
    dark, _ = load_processing(dark_id_topo_set[i], file_indices = None, crop=None)
    topo = topo - dark
    topos.append(topo)

topos = np.stack(topos)
topo = np.mean(topos,axis=0)
    
# Plot
fig, ax = cimshow(topos)
ax.set_title("Topo")

# Load holograms for stitching

In [ ]:
# Define scan ids for each image
im_id_stitch_set = [2935]
topo_id_stitch_set = [2933]

dark_id_im = [2931]
dark_id_topo = dark_id_im

print("Image Id: %s" % im_id_stitch_set)
print("Topo Id: %s" % topo_id_stitch_set)

## Load image of interest

In [ ]:
# Load image
image_stitches = []
for i,im_id_stitch in enumerate(im_id_stitch_set):
    image_stitch, _ = load_processing(im_id_stitch, file_indices = file_indices_im, crop=None)
    dark, _ = load_processing(dark_id_im[i], file_indices = None, crop=None)
    image_stitch = image_stitch - dark
    image_stitches.append(image_stitch)
    
image_stitches = np.array(image_stitches)
image_stitch = np.mean(image_stitches,axis=0)
    
# Plot
fig, ax = cimshow(image_stitches)
ax.set_title("Image for stitching")

## Load topo data set and average

In [ ]:
# Load topo
topo_stitches = []
for i, topo_id_stitch in enumerate(topo_id_stitch_set):
    topo_stitch, _ = load_processing(topo_id_stitch, file_indices = file_indices_topo, crop=None)
    dark, _ = load_processing(dark_id_topo[i], file_indices = None, crop=None)
    topo_stitch = topo_stitch - dark
    topo_stitches.append(topo_stitch)

topo_stitches = np.array(topo_stitches)
topo_stitch = np.mean(topo_stitches,axis=0)
    
# Plot
fig, ax = cimshow(topo_stitches)
ax.set_title("Topo for stitching")

In [ ]:
# Show comparision between data for stitching and normal holograms
fig, ax = plt.subplots(2, 2, figsize=(8, 8), sharex=True, sharey=True)
mi, ma = np.percentile(image, [0.1, 99.9])
ax[0, 0].imshow(image, vmin=mi, vmax=ma)
ax[0, 0].set_title("Image")
mi, ma = np.percentile(image_stitch, [0.01, 99.9])
ax[0, 1].imshow(image_stitch, vmin=mi, vmax=ma)
ax[0, 1].set_title("Image for stitching")
mi, ma = np.percentile(topo, [0.1, 99.9])
ax[1, 0].imshow(topo, vmin=mi, vmax=ma)
ax[1, 0].set_title("Topo")
mi, ma = np.percentile(topo_stitch, [0.01, 99.9])
ax[1, 1].imshow(topo_stitch, vmin=mi, vmax=ma)
ax[1, 1].set_title("Topo for stitching")

# Center holograms

* Find center of the hologram to get a well-defined q-space. 
* Create smooth mask for beamstop or overexposed areas in direct beam

## Basic widget to find center

Try to **align** the circles to the **center of the scattering pattern**. Care! Position of beamstop might be misleading and not represent the actual center of the hologram. 

In [ ]:
# Find center position via widget
c0, c1 = [946, 1040]  # initial values
c0, c1 = [654, 668]  # initial values
#c0, c1 = [258, 244]  # initial values
ic = interactive.InteractiveCenter(topo, c0=c0, c1=c1)

In [ ]:
# Get center positions
center = [ic.c0, ic.c1]
print(f"Center:", center)

## Azimuthal integrator widget for finetuning
More of an "expert widget" which works very well for alignment if you have an Airy Pattern as a scattering image. Transform image from carthesian into polar coordinate system with angle `phi` and radial distance `q` as axis (Azimuthal transformation). If the center is set correctly, all rings of the Airy pattern will be transformed into a straight line spanning of phi at a given q.  

In [ ]:
# Setup azimuthal integrator for virtual geometry
ai = AzimuthalIntegrator(
    dist=experimental_setup["ccd_dist"],
    detector=detector,
    wavelength=experimental_setup["lambda"],
    poni1=center[0]
    * experimental_setup["px_size"]
    * experimental_setup["binning"],  # y (vertical)
    poni2=center[1]
    * experimental_setup["px_size"]
    * experimental_setup["binning"],  # x (horizontal)
)

In [ ]:
# Not the widget, just for double checking to find correct radial range for plotting
q_range_plotting = (0.03, 0.15)

# Perform azimuthal transformation
I_t, q_t, phi_t = ai.integrate2d(
    helper.log_clip(topo),
    500,  # number of points for phi
    radial_range=q_range_plotting,  # relevant q-range
    unit="q_nm^-1",
    correctSolidAngle=False,
    method="BBox",
)
# Combine in an xarray for plotting
az2d = xr.DataArray(I_t, dims=("phi", "q"), coords={"q": q_t, "phi": phi_t})

# Plot
fig, ax = plt.subplots()
mi, ma = np.percentile(I_t, [1, 95])
az2d.plot.imshow(ax=ax, vmin=mi, vmax=ma)
plt.title(f"Azimuthal integration")

In [ ]:
# The widget
aic = interactive.AzimuthalIntegrationCenter(
    helper.log_clip(topo),
    # image,
    ai,
    c0=center[0],
    c1=center[1],
    im_data_range=[1, 98],
    radial_range=q_range_plotting,
    qlines=[100, 110],
)

In [ ]:
# Get center positions from widget
center = [aic.c0, aic.c1]
print(f"Center:", center)

## Center image hologram

In [ ]:
# Apply to topo and image
shift_c = np.array(image.shape) / 2 - center
im_c = cci.shift_image(image, shift_c)
topo_c = cci.shift_image(topo, shift_c)  # centered image

## Center Stitching holograms

In [ ]:
image_stitch_c = cci.shift_image(image_stitch, shift_c)  # centered image
topo_stitch_c = cci.shift_image(topo_stitch, shift_c)  # centered image

# Create beamstops

We want to cover the beamstop with a smooth circle to cover its sharp edges as these would create ringing-like artifacts in the reconstruction plane. Make it only as large as necessary to keep as much information as possible.

## Manual masking

In [ ]:
poly_mask = interactive.draw_polygon_mask(helper.log_clip(im_c))

In [ ]:
# Take poly coordinates and mask from widget
p_coord = poly_mask.get_vertice_coordinates()
mask_draw = poly_mask.full_mask.astype(int)

print("Copy these coordinates into the 'load_poly_coordinates()' function:")
print(p_coord)

# Plot image with beamstop and valid pixel mask
fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(9, 3))
mi, ma = np.percentile(im_c * (1 - mask_draw), [0.1, 99.9])
ax[0].imshow(im_c * (1 - mask_draw), cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Image * (1-mask_draw)")

mi, ma = np.percentile(im_c * mask_draw, [0.1, 99.9])
ax[1].imshow(im_c * mask_draw, vmin=mi, vmax=ma)
ax[1].set_title("Image * mask_draw")

ax[2].imshow(1 - mask_draw)
ax[2].set_title("1 - mask_draw")
plt.tight_layout()

In [ ]:
def load_poly_coordinates():
    """
    Dictionary that stores polygon corner coordinates of all drawn masks
    Example: How to add masks with name "test":
    mask_coordinates["test"] = copy coordinates from above
    """
    mask_coordinates = dict()
    mask_coordinates["bs_fixed"] = [[(988.2, 990.2), (984.2, 994.4), (981.3, 998.7), (977.6, 1005.4), (975.0, 1012.4), (974.1, 1019.8), (973.8, 1027.4), (974.8, 1036.8), (978.6, 1045.5), (985.1, 1055.4), (994.3, 1063.7), (1008.0, 1070.0), (1017.1, 1071.3), (1030.1, 1070.7), (1040.0, 1068.9), (1046.8, 1063.9), (1055.5, 1056.5), (1061.6, 1049.8), (1066.1, 1040.2), (1068.3, 1026.0), (1067.6, 1014.4), (1063.3, 1001.6), (1054.6, 989.9), (1039.8, 980.1), (1026.3, 977.4), (1010.9, 977.6), (1000.5, 981.2), (992.5, 986.5)], [(998.8, 983.3), (982.8, 952.5), (972.0, 934.3), (972.7, 930.4), (971.3, 925.6), (967.4, 923.3), (964.2, 917.6), (964.1, 905.3), (957.5, 902.2), (954.5, 898.7), (954.5, 892.1), (948.6, 886.9), (946.5, 885.6), (946.5, 879.6), (943.0, 875.5), (920.2, 835.4), (883.5, 766.0), (830.6, 666.8), (780.8, 573.0), (742.5, 502.3), (738.2, 505.0), (777.5, 577.0), (787.8, 595.5), (825.3, 667.0), (849.8, 713.1), (878.5, 766.6), (892.3, 792.3), (917.7, 840.1), (935.9, 876.1), (935.4, 878.6), (934.6, 884.1), (937.0, 886.7), (941.8, 886.0), (943.5, 888.9), (941.9, 893.4), (943.5, 898.5), (947.2, 899.9), (948.6, 900.9), (951.2, 905.4), (950.8, 909.8), (951.2, 915.5), (955.0, 918.6), (957.9, 919.3), (962.7, 925.8), (962.7, 929.6), (963.2, 935.0), (968.1, 936.2), (979.4, 959.8), (995.1, 988.5)], [(1036.9, 1066.7), (1040.3, 1073.7), (1040.1, 1077.6), (1041.0, 1080.0), (1044.9, 1081.2), (1058.7, 1107.6), (1058.7, 1112.4), (1062.8, 1115.3), (1067.9, 1125.5), (1079.7, 1146.5), (1080.2, 1151.6), (1082.4, 1153.8), (1086.0, 1157.7), (1086.8, 1163.5), (1089.7, 1165.2), (1092.3, 1171.5), (1094.0, 1176.3), (1096.4, 1178.0), (1100.6, 1187.2), (1102.7, 1193.0), (1106.4, 1198.3), (1116.5, 1216.7), (1117.0, 1223.0), (1123.3, 1230.3), (1128.0, 1238.8), (1128.0, 1243.6), (1133.3, 1246.0), (1137.4, 1255.8), (1143.0, 1266.0), (1153.3, 1284.7), (1153.3, 1288.6), (1154.3, 1291.7), (1158.8, 1295.1), (1161.3, 1300.8), (1184.9, 1344.8), (1209.4, 1390.3), (1225.3, 1420.6), (1226.0, 1425.9), (1241.2, 1447.6), (1260.1, 1485.8), (1313.3, 1586.8), (1337.1, 1632.3), (1342.1, 1630.4), (1289.7, 1528.6), (1246.2, 1445.2), (1234.1, 1426.0), (1233.9, 1420.2), (1166.8, 1296.7), (1167.3, 1290.0), (1162.7, 1286.1), (1158.2, 1280.8), (1137.2, 1243.4), (1137.6, 1237.9), (1133.8, 1235.4), (1125.6, 1221.2), (1125.6, 1215.0), (1121.8, 1212.8), (1104.4, 1180.1), (1098.3, 1169.5), (1094.4, 1163.5), (1094.9, 1160.6), (1091.3, 1155.8), (1067.9, 1112.9), (1067.9, 1108.4), (1064.8, 1105.0), (1050.1, 1078.8), (1050.1, 1074.4), (1045.6, 1068.9), (1041.5, 1064.6)]]
    mask_coordinates["cmos_damage"] = [[(1117.3, 1126.5), (1106.3, 1135.0), (1105.8, 1144.3), (1111.1, 1150.8), (1122.4, 1153.6), (1127.2, 1148.0), (1126.9, 1135.6), (1122.1, 1129.4)]]
    mask_coordinates["bs_large"] = [[(700.0, 30.2), (704.0, -5.6), (728.7, -4.3), (713.9, 222.8), (683.1, 593.1), (685.6, 599.8), (718.6, 629.5), (721.1, 648.9), (717.7, 670.8), (705.6, 692.1), (684.3, 706.9), (672.3, 709.4), (667.1, 768.3), (659.7, 860.9), (642.1, 1098.8), (626.0, 1311.4), (593.3, 1309.6), (615.8, 1109.0), (638.7, 817.1), (645.8, 708.2), (625.1, 694.6), (614.6, 681.9), (609.4, 668.7), (606.9, 659.4), (592.4, 660.9), (575.7, 659.7), (573.6, 636.6), (592.7, 637.5), (610.3, 637.2), (618.3, 623.0), (636.8, 606.0), (655.3, 597.4), (671.1, 437.8), (685.9, 209.8)]]
    mask_coordinates["bs_fixed_scaled"] = [[(648.8, 321.7), (658.0, 620.8), (642.7, 636.9), (640.4, 657.6), (651.9, 675.3), (656.5, 677.6), (667.2, 1082.5), (672.6, 1082.5), (671.1, 676.8), (685.7, 673.7), (697.2, 661.5), (697.2, 644.6), (688.0, 625.4), (668.8, 620.8), (657.0, 321.5)]]
    mask_coordinates["bs_bar"] = [[(662.0, 622.0), (656.8, 623.5), (650.0, 629.2), (645.1, 635.1), (641.9, 644.2), (641.3, 650.3), (643.6, 662.1), (647.0, 667.6), (653.7, 673.1), (657.2, 675.8), (657.2, 682.8), (655.9, 692.0), (654.9, 723.6), (652.0, 757.5), (649.0, 796.3), (648.3, 809.4), (647.0, 831.8), (644.1, 861.6), (640.9, 907.7), (637.8, 939.7), (634.8, 967.1), (631.3, 1004.7), (630.6, 1016.7), (630.4, 1023.2), (629.8, 1032.7), (626.1, 1034.8), (614.9, 1038.5), (608.0, 1040.7), (598.9, 1048.9), (590.3, 1059.6), (583.6, 1070.0), (581.7, 1075.0), (575.6, 1075.3), (548.0, 1074.6), (548.9, 1096.1), (578.4, 1095.2), (581.2, 1103.4), (584.7, 1110.7), (590.9, 1123.4), (598.3, 1130.6), (608.6, 1137.0), (619.0, 1142.4), (618.0, 1174.7), (615.4, 1195.4), (614.3, 1218.2), (612.4, 1242.0), (609.4, 1280.0), (607.9, 1300.7), (627.9, 1301.5), (630.7, 1258.8), (636.6, 1189.1), (641.3, 1144.0), (644.5, 1141.9), (655.3, 1138.6), (663.5, 1135.2), (675.8, 1124.4), (683.8, 1115.2), (688.3, 1101.0), (688.3, 1082.2), (685.7, 1070.7), (678.2, 1056.9), (668.5, 1045.9), (660.5, 1041.0), (651.8, 1037.3), (651.0, 1032.6), (656.2, 952.3), (662.3, 867.2), (669.0, 781.5), (672.8, 714.1), (676.1, 677.0), (680.5, 675.1), (685.1, 673.4), (689.2, 669.3), (694.6, 662.1), (696.7, 652.5), (696.0, 639.4), (691.2, 632.4), (686.9, 628.0), (682.0, 623.9), (681.0, 611.4), (687.7, 523.6), (695.5, 420.2), (703.6, 313.8), (711.0, 216.5), (721.1, 95.1), (729.6, -5.7), (708.7, -3.5), (701.8, 91.7), (699.9, 108.5), (697.1, 144.8), (692.0, 208.5), (688.7, 263.1), (682.4, 329.7), (678.6, 396.1), (675.0, 440.2), (672.7, 469.0), (670.3, 501.8), (668.3, 536.3), (665.9, 562.5), (664.0, 585.6), (662.7, 605.1)]]
    mask_coordinates["bs_fixed_support"] = [[(658.9, 542.9), (658.1, 555.1), (658.7, 563.7), (658.1, 571.9), (657.5, 580.4), (658.3, 583.8), (658.3, 590.2), (658.9, 594.4), (662.5, 593.6), (662.9, 585.6), (662.5, 579.6), (664.1, 576.4), (662.7, 570.9), (662.1, 562.7)]]
    mask_coordinates["bs_fixed_ccd"] = [[(656.7, 625.8), (648.2, 632.9), (642.0, 643.8), (642.6, 654.6), (646.4, 664.7), (660.2, 675.8), (661.9, 699.2), (661.2, 709.9), (662.3, 976.0), (663.6, 1191.2), (665.3, 1318.9), (676.2, 1320.4), (671.5, 786.5), (667.2, 699.5), (666.5, 678.6), (677.5, 676.7), (688.2, 671.7), (695.3, 662.2), (697.9, 647.4), (692.0, 632.9), (682.7, 624.9), (667.7, 621.3), (664.4, 622.0), (663.2, 581.1), (667.0, 577.3), (666.5, 570.7), (663.2, 569.5), (662.0, 313.0), (663.2, -12.4), (651.5, -13.3), (658.0, 560.7), (655.1, 566.9), (657.7, 569.2), (658.0, 572.4), (655.9, 576.4), (657.4, 581.1), (660.1, 620.5)]]
    mask_coordinates["bs_large_ccd"] = [[(707.4, -5.1), (683.6, 337.2), (672.6, 476.4), (663.5, 569.7), (655.7, 574.0), (659.3, 592.5), (660.4, 595.2), (650.8, 598.5), (639.7, 603.6), (628.3, 611.7), (622.3, 622.5), (616.9, 630.6), (615.4, 633.8), (608.8, 634.4), (592.6, 635.0), (587.5, 633.2), (582.1, 634.1), (581.2, 637.4), (580.6, 649.1), (582.1, 655.1), (594.7, 655.7), (605.5, 654.5), (610.3, 655.1), (613.3, 661.1), (616.9, 670.4), (621.4, 680.0), (630.4, 691.1), (638.8, 696.2), (649.9, 702.5), (652.2, 707.5), (647.4, 759.3), (640.8, 848.4), (632.4, 956.2), (622.3, 1080.4), (611.3, 1203.5), (605.3, 1284.4), (604.1, 1305.9), (624.4, 1305.6), (635.1, 1166.3), (648.3, 1021.7), (660.3, 852.1), (671.2, 716.5), (672.4, 705.0), (675.4, 702.0), (685.0, 699.6), (694.9, 695.1), (705.1, 688.5), (711.4, 678.8), (720.1, 662.6), (721.3, 644.3), (720.1, 628.1), (711.7, 614.3), (699.4, 604.1), (690.7, 599.0), (684.4, 596.0), (683.0, 589.4), (687.8, 523.5), (693.2, 451.5), (702.7, 330.8), (711.2, 232.2), (722.6, 70.1), (727.5, -4.7)]]
    mask_coordinates["slit"] = [[(745.0, 571.5), (733.1, 570.3), (732.4, 603.0), (728.3, 652.1), (724.6, 698.9), (721.2, 761.4), (727.6, 760.3), (731.3, 698.9), (735.2, 651.9), (739.4, 622.7)]]
    mask_coordinates["high_q_slit"] = [[(219.9, 532.2), (207.5, 531.7), (207.0, 565.5), (203.0, 625.0), (200.0, 668.2), (193.6, 757.1), (202.0, 757.6), (206.0, 687.6), (206.5, 659.3), (213.4, 600.7), (219.4, 558.0)]]
    return mask_coordinates

## Create masks for image and topo

In [ ]:
# Which drawn masks do you want to load? You can combine multiple masks from
# load_poly_coordinates(). Just add names of mask as strings to list like
# ["bs_small","bs_medium"]
polygon_names = ["bs_fixed_ccd"]
mask_draw = mask_lib.load_poly_masks(
    experimental_setup["binning"] * image.shape,
    load_poly_coordinates(),
    polygon_names,
)

#mask_draw = cci.shift_image(mask_draw,[0,-8])

# Plot image with beamstop and valid pixel mask
fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(9, 3))
mi, ma = np.percentile(im_c * (1 - mask_draw), [0.1, 99.9])
ax[0].imshow(im_c * (1 - mask_draw), cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Image * (1-mask_draw)")

# mi, ma = np.percentile(im_c * mask_draw, [0.1, 90])
ax[1].imshow(im_c * mask_draw, vmin=mi, vmax=ma)
ax[1].set_title("Image * mask_draw")

ax[2].imshow(1 - mask_draw)
ax[2].set_title("1 - mask_draw")

### Finetuning of mask position

In [ ]:
# Use widget to shift and expand or shrink the mask
ss_mask = interactive.Shift_Scale_Mask(im_c, mask_draw, shift=[0,0], scale=0)

In [ ]:
# Take mask, shift and scaling from widget
mask_draw, mask_shift, mask_scale = ss_mask.get_mask()

## Create mask for Stitching

In [ ]:
# Which drawn masks do you want to load for stitching?
polygon_names = ["bs_large_ccd"]
mask_draw_stitch = mask_lib.load_poly_masks(
    experimental_setup["binning"] * image.shape,
    load_poly_coordinates(),
    polygon_names,
)

# Plot image with beamstop and valid pixel mask
fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(9, 3))
mi, ma = np.percentile(image_stitch_c * (1 - mask_draw_stitch), [0.1, 99.9])
ax[0].imshow(image_stitch_c * (1 - mask_draw_stitch), cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Image * (1-mask_draw_stitch)")

mi, ma = np.percentile(image_stitch_c * mask_draw_stitch, [0.1, 99.9])
ax[1].imshow(image_stitch_c * mask_draw_stitch, vmin=mi, vmax=ma)
ax[1].set_title("Image * mask_draw_stitch")

ax[2].imshow(1 - mask_draw_stitch)
ax[2].set_title("1 - mask_draw_stitch")
plt.tight_layout()

## Finetuning of mask position

In [ ]:
# Use widget to shift and expand or shrink the mask
ss_mask = interactive.Shift_Scale_Mask(image_stitch_c, mask_draw_stitch, shift=[0,0], scale=0)

In [ ]:
# Take mask, shift and scaling from widget
mask_draw_stitch, mask_shift, mask_scale = ss_mask.get_mask()

# Image Registration (not established yet)

Relative drift between data holograms and their corresponding topo holograms is calculated by image registration algorithm. Necessary to get well defined difference hologram. The reference is always the static background image (topo).

## Set Alignment ROI 

Set a region of interest (ROI) of reference (topo) use for image registration is performed. Don't select ROI which contains the beamstop as this leads to wrong results

How to use:
1. Zoom into the image and adjust your FOV until you are satisfied.
2. Save the axes coordinates.

In [ ]:
fig, ax = cimshow(im_c * (1 - mask_draw))
ax.set_title("Can include beamstop")

In [ ]:
# Takes start and end of x and y axis
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi_im_reg = np.array([y1, y2, x1, x2]).astype(int)
roi_im_reg = np.s_[roi_im_reg[0] : roi_im_reg[1], roi_im_reg[2] : roi_im_reg[3]]

print(f"Image registration roi:", roi_im_reg)

## Calculate drift of images

In [ ]:
shift = cci.image_registration(
    im_c[roi_im_reg],
    topo_c[roi_im_reg],
    method="dipy",
    #static_mask=mask_draw[roi_im_reg],
    #moving_mask=mask_draw[roi_im_reg],
)
print(shift)

In [ ]:
shift = [0, 0]

## Center and apply circular beamstop to topo image

In [ ]:
# Shift and apply beamstop
topo_c = cci.shift_image(topo_c, shift)  # centered image
topo_stitch_c = cci.shift_image(topo_stitch_c, shift)

# Plot original and shifted holos
mi, ma = np.percentile(np.real(im_c[im_c != 0]), (0.1, 99.9))
fig, ax = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(8, 4))
ax[0].imshow(np.real(image), cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Uncentered image")
ax[1].imshow(np.real(im_c), cmap="viridis", vmin=mi, vmax=ma)
ax[1].set_title("Centered image with beamstop")

# Add circles with different radi r
tmp = np.array(image.shape) / 2
for r in np.arange(50, 200, 50):
    ax[0].add_artist(plt.Circle((tmp[1], tmp[0]), r, fill=None, ec="red"))
    ax[1].add_artist(plt.Circle((tmp[1], tmp[0]), r, fill=None, ec="red"))

# Execute Stitching

## Define Rois to calc scaling between holograms

In [ ]:
# Define mask for each image
mask_im = mask_draw.copy()
mask_stitch = mask_draw_stitch.copy()

mask_both = mask_im + mask_stitch
mask_both[mask_both>1] = 1 

In [ ]:
tmp = np.stack([im_c,image_stitch_c])
fig, ax = cimshow(helper.log_clip(tmp))
ax.set_title("Choose roi with relevant statistics for scaling")
ax.imshow(mask_both, alpha=0.3)

In [ ]:
# Takes start and end of x and y axis
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi_stitch = np.array([y1, y2, x1, x2]).astype(int)
roi_stitch = np.s_[roi_stitch[0] : roi_stitch[1], roi_stitch[2] : roi_stitch[3]]
print(f"Stitching roi:", roi_stitch)

## Calc scaling

### Define additional masks

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(image_stitch_c[(mask_both).astype(bool)], 300)
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_title("Stitch Image")
ax[1].hist(topo_stitch_c[(mask_both).astype(bool)], 300)
ax[1].set_title("Stitch Topo")
ax[1].set_xscale("log")
ax[1].set_yscale("log")

In [ ]:
# Show comparision between data for stitching and normal holograms
fig, ax = plt.subplots(2, 2, figsize=(8, 8), sharex=True, sharey=True)
mi, ma = np.percentile((im_c * (1 - mask_both))[roi_stitch], [0.1, 99.99])
ax[0, 0].imshow((im_c * (1 - mask_both))[roi_stitch], vmin=mi, vmax=ma)
ax[0, 0].set_title("Image")
mi, ma = np.percentile((image_stitch_c * (1 - mask_both))[roi_stitch], [0.1, 99.99])
ax[0, 1].imshow((image_stitch_c * (1 - mask_both))[roi_stitch], vmin=mi, vmax=ma)
ax[0, 1].set_title("Image for stitching")


mi, ma = np.percentile((topo_c * (1 - mask_both))[roi_stitch], [0.1, 99.99])
ax[1, 0].imshow((topo_c * (1 - mask_both))[roi_stitch], vmin=mi, vmax=ma)
ax[1, 0].set_title("Topo")
mi, ma = np.percentile((topo_stitch_c * (1 - mask_both))[roi_stitch], [0.1, 99.99])
ax[1, 1].imshow((topo_stitch_c * (1 - mask_both))[roi_stitch], vmin=mi, vmax=ma)
ax[1, 1].set_title("Topo for stitching")

### Fitting

In [ ]:
# Get scaling factor and offset of image
back_im_c, _ = np.percentile(im_c, [0.1, 100])
back_im_stitch, _ = np.percentile(image_stitch_c, [1, 100])

factor_stitch_im, offset_stitch_im = cci.dyn_factor(
    ((im_c - back_im_c) * (1 - mask_both))[roi_stitch],
    ((image_stitch_c - back_im_stitch) * (1 - mask_both))[roi_stitch],
    method="correlation",
    verbose=True,
    plot=True,
)

# Get scaling factor and offset of topo
back_topo_c, _ = np.percentile(topo_c, [0.1, 100])
back_topo_stitch, _ = np.percentile(topo_stitch_c, [10, 100])

factor_stitch_topo, offset_stitch_topo = cci.dyn_factor(
    ((topo_c - back_topo_c) * (1 - mask_both))[roi_stitch],
    ((topo_stitch_c - back_topo_stitch) * (1 - mask_both))[roi_stitch],
    method="correlation",
    verbose=True,
    plot=True,
)

print(
    "Backgrounds im: %.1f im_stitch: %.1f topo: %.1f topo_stitch: %.1f"
    % (back_im_c, back_im_stitch, back_topo_c, back_topo_stitch)
)

## Perform stitching

In [ ]:
# Zoom into transition region for plotting
fig, ax = cimshow(im_c)

In [ ]:
roi = interactive.axis_to_roi(ax)

In [ ]:
correction = 1  # 0.5

# Stitch together
im_stitched = (
    im_c * (1 - mask_im)
    + (
        correction * factor_stitch_im * (image_stitch_c - back_im_stitch)
        + back_im_stitch
    )
    * mask_im
)
topo_stitched = (
    topo_c * (1 - mask_im)
    + (
        correction * factor_stitch_topo * (topo_stitch_c - back_topo_stitch)
        + back_topo_stitch
    )
    * mask_im
)

#mask_stitch = mask_im.copy()

# Plot
fig, ax = plt.subplots(2, 3, figsize=(12, 8), sharex=True, sharey=True)
mi, ma = np.percentile(image_stitch_c[roi], [0.1, 99.99])
ax[0, 0].imshow(image_stitch_c[roi], vmin=mi, vmax=ma)
ax[0, 0].imshow(mask_stitch[roi], alpha=0.2)
ax[0, 0].set_title("Centered Stitching Image")
mi, ma = np.percentile(im_c[roi], [0.1, 99.99])
ax[0, 1].imshow(im_c[roi], vmin=mi, vmax=ma)
ax[0, 1].imshow(mask_stitch[roi], alpha=0.2)
ax[0, 1].set_title("Centered Image")
mi, ma = np.percentile(im_stitched[roi], [0.1, 99])
ax[0, 2].imshow(im_stitched[roi], vmin=mi, vmax=ma)
# ax[0, 2].imshow(mask_stitch, alpha=0.2)
ax[0, 2].set_title("Stitched Image")

mi, ma = np.percentile(topo_stitch_c[roi], [0.1, 99.99])
ax[1, 0].imshow(topo_stitch_c[roi], vmin=mi, vmax=ma)
# ax[1, 0].imshow(mask_stitch, alpha=0.2)
ax[1, 0].set_title("Centered Stitching Topo")
mi, ma = np.percentile(topo_c[roi], [0.1, 99.9])
ax[1, 1].imshow(topo_c[roi], vmin=mi, vmax=ma)
ax[1, 1].imshow(mask_stitch[roi], alpha=0.2)
ax[1, 1].set_title("Centered Topo")
mi, ma = np.percentile(topo_stitched[roi], [0.1, 99])
ax[1, 2].imshow(topo_stitched[roi], vmin=mi, vmax=ma)
# ax[1, 2].imshow(mask_stitch, alpha=0.2)
ax[1, 2].set_title("Stitched Topo")

## Add circles with different radi r
# tmp = np.array(image.shape) / 2
# for r in np.arange(0, 5, 10):
#   ax[0, 0].add_artist(plt.Circle((tmp[1], tmp[0]), r, fill=None, ec="red"))

In [ ]:
#cimshow(helper.log_clip(im_stitched))
#cimshow(im_stitched)

#cimshow(helper.log_clip(topo_stitched))
cimshow(topo_stitched)

# Calculate difference holograms

You can see the reconstrution of the magnetization only after subtracting the large background that you get from the diffraction on the circular object aperture (Airy Pattern). This might require a scaling factor to correct intensity changes between the hologram and the topo. Scaling factor will be determined automatically by a linear fit. If the fit seems off, there might be an issue with the data

# Create beamstops

We want to cover the beamstop with a smooth circle to cover its sharp edges as these would create ringing-like artifacts in the reconstruction plane. Make it only as large as necessary to keep as much information as possible.

In [ ]:
# Create combined beamstop
mask_pixel = mask_im * mask_stitch

# Create smooth mask
footprint = skimage.morphology.disk(6)
mask_pixel_smooth = skimage.morphology.dilation(mask_pixel, footprint)
mask_pixel_smooth = gaussian_filter(mask_pixel_smooth, 2)

# Plot both
fig, ax = plt.subplots(2, 4, figsize=(10, 5), sharex=True, sharey=True)
mi, ma = np.percentile(im_c, [1, 99.9])
ax[0, 0].imshow(im_stitched, vmin=mi, vmax=ma)
ax[0, 0].set_title("Image")
mi, ma = np.percentile(im_stitched * mask_pixel, [1, 99.99])
ax[0, 1].imshow(im_stitched * mask_pixel, vmin=mi, vmax=ma)
ax[0, 1].set_title("Image*mask")
mi, ma = np.percentile(im_stitched * (1 - mask_pixel), [0.1, 99.9])
ax[0, 2].imshow(im_stitched * (1 - mask_pixel), vmin=mi, vmax=ma)
ax[0, 2].set_title("Image*(1-mask)")
ax[0, 3].imshow(mask_pixel_smooth)
ax[0, 3].set_title("Combined Mask")

mi, ma = np.percentile(topo_c, [1, 99.9])
ax[1, 0].imshow(topo_stitched, vmin=mi, vmax=ma)
ax[1, 0].set_title("Topo")
mi, ma = np.percentile(topo_stitched * mask_pixel, [1, 99.99])
ax[1, 1].imshow(topo_stitched * mask_pixel, vmin=mi, vmax=ma)
ax[1, 1].set_title("Topo*mask")
mi, ma = np.percentile(topo_stitched * (1 - mask_pixel), [0.1, 99.9])
ax[1, 2].imshow(topo_stitched * (1 - mask_pixel), vmin=mi, vmax=ma)
ax[1, 2].set_title("topo*(1-mask)")
mi, ma = np.percentile((im_stitched - topo_stitched) * (1 - mask_pixel_smooth), [0.1, 99.9])
ax[1, 3].imshow((im_stitched - topo_stitched) * (1 - mask_pixel_smooth), vmin=mi, vmax=ma)
ax[1, 3].set_title("Image-Topo")

In [ ]:
# Get scaling factor and offset
factor, offset = cci.dyn_factor(
    im_c * (1 - mask_pixel),
    topo_c * (1 - mask_pixel),
    method="correlation",
    verbose=False,
    plot=True,
)

# Calculate differences (magnetic) and sums (topographc) contrast holograms.
# _c: centered, without beamstop, _b: centered, with beamstop
diff_c = im_c / factor - topo_c - offset
sum_c = im_c / factor + topo_c - offset

# Get scaling factor and offset
factor, offset = cci.dyn_factor(
    im_stitched,
    topo_stitched,
    method="correlation",
    verbose=False,
    plot=True,
)

diff_stitched = im_stitched/ factor - topo_stitched - offset
sum_stitched = im_stitched/ factor + topo_stitched - offset

In [ ]:
# Plot an example of the difference or sum hologram
fig, ax = cimshow(diff_stitched)
ax.set_title(f" Diff Id %d" % im_id)

# fig, ax = cimshow(sum_b)
# ax.set_title(f" Sum Id %d" % im_id)

In [ ]:
tmp = np.sign(diff_stitched)*np.log10(np.abs(diff_stitched.copy()))
cimshow(tmp)

# Reconstruct Diff Holos (FTH)

Reconstruct the hologramm.
1. Chose a region of interest (ROI) which means selecting one reconstruction from the rconstruction plane.
2. Propagate the image and shift the phase for maximal contrast and sharpness in your ROI
3. Optional finetuning with a widget

### Set Patterson Map ROI

Choose the reconstructions as the ROI.

1. Zoom into the image and adjust your FOV until you are satisfied.
2. Save the axes coordinates.

In [ ]:
# Choose contrast mode
# diff_c: magnetic contrast only
# sum_c: topographic contrast only

# Original images
holo = diff_c * (1 - mask_pixel_smooth)
#holo = im_c * (1 - mask_pixel_smooth)
#holo = sum_c * (1 - mask_pixel_smooth)

# Stitched images
holo = diff_stitched * (1 - mask_pixel_smooth)
#holo = im_stitched * (1 - mask_pixel_smooth)
#holo = sum_stitched * (1 - mask_pixel_smooth)

# Plotting
tmp = cci.reconstruct(holo)
fig, ax = cimshow(np.real(tmp), cmap="gray")

In [ ]:
# Execute to get roi
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi = np.array([y1, y2, x1, x2]).astype(int)  # ystart, ystop, xstart, xstop
roi = [ 206,  595,  287 ,1068]
roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
print(f"Roi Reco:{roi}")

## Tune propagation and phase
Focus the image by tuning the propagation distance. This really works like focussing in a microscope.
Phase slider will move contrast between real and imaginary part. Usually we use the phase which maximizes the contrast in the real part

In [ ]:
# Widget
slider_prop, slider_phase = interactive.propagate_phase(
    diff_c* (1 - mask_pixel_smooth),
    roi_s,
    experimental_setup=experimental_setup,
    scale=(1, 99),
)

In [ ]:
# Read prop dist and phase from widget
prop_dist = slider_prop.value
phase = slider_phase.value

print(f"Propagation distance: %0.2f" % prop_dist)
print(f"Phase: %0.2f" % phase)

## Save reconstruction

Save png files of the images and a h5 file for all important variables

In [ ]:
# Saves only real and imaginary part
holo = diff_stitched * (1 - mask_pixel_smooth)
recon = np.zeros(image.shape, dtype=np.complex_)

# Reconstruct
recon = cci.reconstruct(
    cci.propagate(holo, prop_dist * 1e-6, experimental_setup=experimental_setup)
    * np.exp(1j * phase)
)

# Plot
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Image %d - %d Stitching" % (im_id, topo_id))

vmin, vmax = np.percentile(np.real(recon[roi_s]), (0.1, 99))
t_im1 = ax[0].imshow(np.real(recon[roi_s]), vmin=vmin, vmax=vmax, cmap="gray")
ax[0].set_title("Real")
plt.colorbar(t_im1, ax=ax[0], aspect=50)

vmin, vmax = np.percentile(np.imag(recon[roi_s]), (0.1, 99))
t_im2 = ax[1].imshow(np.imag(recon[roi_s]), vmin=vmin, vmax=vmax, cmap="gray")
ax[1].set_title("Imag")
plt.colorbar(t_im2, ax=ax[1], aspect=50)

# Save images
fname = join(
    folder_general,
    "Recon_ImId_%04d_RefId_%04d_stitching_%s.png" % (im_id, topo_id, USER),
)
print("Saving: %s" % fname)
plt.savefig(fname, bbox_inches="tight", transparent=False)

# Save hdf5 file
#save_fth_h5()

In [ ]:
# Closes all existing plots
plt.close("all")

# CDI Reconstruction

## Create set of pos and neg helicity holograms

In [ ]:
# Copy values from FTH reco
pos = (sum_stitched + diff_stitched) / 2
neg = (sum_stitched - diff_stitched) / 2

pos = im_stitched/factor
neg = topo_stitched.copy()

## Create Support mask
The support mask is the real-space constraint used for the (holographically-aided) phase retrieval, i.e., certain details about our sample like the sample geometry. For our samples we can directly derive a very strong constraint: The FTH reconstructions show us previsely the actual real-space sample structure, i.e., the arrangement of our aperture where x-rays are transmitted ("1") while the masked areas show no transmission ("0"). We will therefore create a binary mask that reflects this transmission as an input for the phase retrieval.

How to draw a support mask: Create a binary mask of the locations of sample apertures in the fth reconstruction. Areas with apertures are "1". Select only a single set of reconstructions (object & reference apertures) that originate from a single reference. Use the widget!

### Option 1: Execute if you want to create a new support mask
If you really want to create a new support mask, execute next cell and then the "InteractiveCircleCoordinates"-widget

In [ ]:
# How many references do you have?
nr_ref = 5

# Setup coordinates (nr_ref + 1 coordinates, as there is always the object aperture)
support_coordinates = [
    [pos.shape[-2] // 2, pos.shape[-1] // 2, 7] for k in range(nr_ref + 1)
]

# Widget to find the positions and sizes of the different apertures
print(
    "Cover the object & reference apertures for each set of reconstructions that originates from the same reference with circles."
)
print(
    "Optimization: Change one circle parameter, calc phase retrieval image, compare with images reconstructed with old circle parameter. Repeat!"
)

# Create plot
holo = pos * (1 - mask_pixel_smooth)

# Reconstruct
recon = cci.reconstruct(
    cci.propagate(holo, prop_dist * 1e-6, experimental_setup=experimental_setup)
    * np.exp(1j * phase)
)
recon = np.real(recon)

ds = interactive.InteractiveCircleCoordinates(
    recon,
    len(support_coordinates),
    coordinates=support_coordinates.copy(),
)

In [ ]:
# Take coordinates of circles from widget
support_coordinates = ds.get_params()

# Create supportmask from coordinates
supportmask = mask_lib.create_circle_supportmask(support_coordinates, pos.shape)

# Plot supportmask as overlay
fig, ax = plt.subplots(figsize=(6, 6))
mi, ma = np.percentile(recon, (1, 99))
ax.imshow(recon, vmin=mi, vmax=ma, cmap="gray")
ax.imshow(supportmask, alpha=0.4, cmap="binary")
ax.set_title("Image with overlayed mask")

### Option 2: Execute if you want to load an existing support mask created with circle mask widget

In [ ]:
def get_supportmask_coordinates(sample):
    """
    Dictionary that stores coordinates of circular support mask apertures
    """

    # Setup dictonary
    support_coord = dict()

    # coordinates
    support_coord["FB0022_D6"] = [(854.0, 945.0, 47.5), (922.0, 1122.0, 7.0), (779.5, 1119.0, 7.0), (681.5, 1016.5, 7.0), (1024.0, 1024.0, 7.0)]
    support_coord["s2601i_D1"] = [(920.5, 1024.5, 30.5), (952.0, 925.0, 5.0), (952.5, 1123.0, 5.0), (835.5, 1085.5, 5.0), (835.5, 963.0, 5.0), (1024.0, 1024.0, 5.0)]
    #support_coord["s2601i_D1"] = [(939.5, 963.5, 29.5), (1024.0, 901.6, 5.0), (906.5, 864.1, 5.0), (907.5, 1062.0, 5.0), (835.5, 963.0, 5.0), (1024.0, 1024.0, 5.0)]
    support_coord["s2601i_D1_Gd"] = [(882.0, 920.0, 46.0), (1023.0, 820.0, 5.0), (828.5, 756.5, 5.0), (708.0, 921.5, 5.0), (829.0, 1087.0, 5.0), (1024.0, 1024.0, 5.0)]
    support_coord["s2601i_D1_energy_scan"] = [(939.0, 962.0, 29.5), (1024.0, 901.6, 5.0), (906.5, 864.1, 5.0), (907.5, 1062.0, 5.0), (835.5, 963.0, 5.0), (1024.0, 1024.0, 5.0)]
    support_coord["s2602g_B3"] = [(432.0, 398.5, 87.0), (746.0, 266.5, 5.5), (404.5, 50.0, 5.5), (96.0, 314.5, 5.5), (250.0, 700.0, 5.5), (650.0, 670.0, 5.5)]
    support_coord["s2602d_C3"] = [(490.0, 470.0, 70.5), (234.0, 406.0, 9.5), (464.5, 205.5, 9.5), (722.0, 368.5, 9.5), (348.5, 693.0, 9.5), (650.0, 670.0, 9.5)]
    support_coord["s2601f_D7_cropped"] = [(174.0, 158.0, 34.5), (104.9, 266.9, 5.0), (49.0, 124.5, 6.0), (165.0, 29.0, 6.5), (294.5, 108.5, 7.0), (257.0, 256.0, 6.0)]
    support_coord["s2601f_H7_cropped"] = [(174.0, 158.0, 35.0), (104.9, 266.9, 5.0), (49.0, 124.5, 6.0), (165.0, 29.0, 6.5), (294.5, 108.5, 7.0), (257.0, 256.0, 6.0)]
    support_coord["s2601f_H7"] = [(444.0, 412.5, 85.5), (124.0, 327.0, 14.0), (418.5, 75.0, 13.5), (745.0, 286.0, 14.0), (266.0, 695.5, 13.0), (650.0, 670.0, 13.5)]
    support_coord["s2601f_J7"] = [(444.0, 408.0, 85.5), (122.0, 327.5, 14.0), (418.5, 73.5, 13.5), (744.9, 284.5, 14.0), (264.5, 696.0, 13.0), (650.0, 670.0, 13.5)]
    support_coord["s2601h_H7"] = [(440.5, 409.0, 88.5), (119.5, 329.0, 12.0), (415.5, 74.0, 12.0), (743.5, 284.0, 12.0), (264.5, 697.5, 12.0), (650.0, 670.0, 12.0)]
    return support_coord[sample]

In [ ]:
# Which supportmask to load? ("s2306a-C1", "s2308a-B1", ...)
sample = "s2601f_H7"

# Get coordinates and create supportmask
support_coordinates = get_supportmask_coordinates(sample)

In [ ]:
# Widget to find the positions and sizes of the different apertures
print(
    "Cover the object & reference apertures for each set of reconstructions that originates from the same reference with circles."
)
print(
    "Optimization: Change one circle parameter, calc phase retrieval image, compare with images reconstructed with old circle parameter. Repeat!"
)

# Create plot
holo = sum_c * (1 - mask_pixel_smooth)
ds = interactive.InteractiveCircleCoordinates(
    np.real(cci.reconstruct(holo)),
    len(support_coordinates),
    coordinates=support_coordinates,
)

In [ ]:
# Take coordinates of circles from widget
support_coordinates = ds.get_params()

# Create supportmask
supportmask = mask_lib.create_circle_supportmask(
    support_coordinates,pos.shape
)

# What to plot?
tmp = np.real(cci.reconstruct(holo))

# Plot supportmask as overlay
fig, ax = plt.subplots(figsize=(6, 6))
mi, ma = np.percentile(tmp, (1, 99))
ax.imshow(tmp, vmin=mi, vmax=ma, cmap="gray")
ax.imshow(supportmask, alpha=0.3, cmap="binary")
ax.set_title("Image with overlayed mask")

### Take Roi

In [ ]:
fig, ax = cimshow(supportmask.astype(int))

In [ ]:
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi_cdi = np.array([int(y1), int(y2), int(x1), int(x2)])  # xstart, xstop, ystart, ystop

roi_cdi = [337, 553, 296, 533]
roi_cdi_s = np.s_[roi_cdi[0] : roi_cdi[1], roi_cdi[2] : roi_cdi[3]]
print("Sliced roi:", roi_cdi)

## Do Phase Retrieval

In [ ]:
# Define your recipe for the phase retrieval process. Undefined parameter are taken from default settings
phase_retrieval_recipe = dict()
phase_retrieval_recipe["hologram_intensity_cutoff_vmin"] = 0.1
phase_retrieval_recipe["algorithm_list_full_coherence "] = ["HAPRE","ER","ER"]
phase_retrieval_recipe["algorithm_list_partial_coherence "] = ["HAPRE","ER","ER"]
phase_retrieval_recipe["number_iterations_partial_coherence"] = [700,50,50]

In [ ]:
# Executes the algorithm
(
    retrieved_p,
    retrieved_n,
    retrieved_p_pc,
    retrieved_n_pc,
    bsmask_p,
    bsmask_n,
    gamma_p,
    gamma_n,
) = PhR.phase_retrieval_algorithm(
    pos,
    neg,
    mask_pixel,
    supportmask,
    phase_retrieval_recipe=phase_retrieval_recipe,
)

## Reconstruct images

In [ ]:
# New beamstop for CDI recos as phase retrieval of low-q might be insufficient. If phase retrieval worked well
# Try without beamstop: `use_bs = False`
use_bs = False
bs_diam_cdi = 25  # diameter of beamstop

# Create beamstop
if use_bs is True:
    mask_bs_cdi = 1 - mask_lib.circle_mask(
        topo.shape, np.array(topo.shape) / 2, bs_diam_cdi, sigma=4
    )
    mask_bs_cdi = 1 - mask_pixel_smooth.copy()
elif use_bs is False:
    mask_bs_cdi = np.ones(pos.shape)  # if you don't want a beamstop

# Plotting
mode = "-"
print("Fine-tuning of reconstruction parameter:")
slider_prop, slider_phase, slider_dx, slider_dy = interactive.focusCDI(
    retrieved_p_pc * mask_bs_cdi,
    retrieved_n_pc * mask_bs_cdi,
    roi_cdi_s,
    mask=supportmask,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
    prop_dist=prop_dist_cdi,
    experimental_setup=experimental_setup,
    operation=mode,
    max_prop_dist=5,
    scale=(3, 97),
)

In [ ]:
# Get phase from slider
phase_cdi = slider_phase.value
prop_dist_cdi = slider_prop.value

# Reconstruct images with new parameter
p_pc = cci.FFT(
    cci.propagate(
        retrieved_p_pc * mask_bs_cdi,
        prop_dist_cdi * 1e-6,
        experimental_setup=experimental_setup,
    )
) * np.exp(1j * phase_cdi)

n_pc = cci.FFT(
    cci.propagate(
        retrieved_n_pc * mask_bs_cdi,
        prop_dist_cdi * 1e-6,
        experimental_setup=experimental_setup,
    )
) * np.exp(1j * phase_cdi)

print("Phase CDI: %s" % phase_cdi)
print("Prop_dist: %s" % prop_dist_cdi)

In [ ]:
# Confirm that offset subtraction works, i.e., only small fraction of hologram is actually masked
fig, ax = plt.subplots(2, 2, figsize=(8, 8), sharex=True, sharey=True)
tmp = np.abs(retrieved_p_pc * mask_bs_cdi)
mi, ma = np.percentile(tmp, [0.1, 99.9])
ax[0, 0].imshow(tmp, vmin=mi, vmax=ma)
ax[0, 0].set_title("Pos holo")

tmp = np.abs(retrieved_n_pc)
mi, ma = np.percentile(tmp, [0.1, 99.9])
ax[0, 1].imshow(tmp, vmin=mi, vmax=ma)
ax[0, 1].set_title("Neg holo")
ax[1, 0].imshow(bsmask_p)
ax[1, 0].set_title("Pos holo mask")
ax[1, 1].imshow(bsmask_n)
ax[1, 1].set_title("Neg holo mask")

## Save reconstructions

In [ ]:
def get_title(data_key, im_id, topo_id, CDI=False, Framenumber= None):
    # Rotation in title
    if data_key is not None:
        data = load_key(im_id,mnemonics[data_key])

    if CDI is False:
        mode = "FTH"
    elif CDI is True:
        mode = "CDI"

    if data_key == "magOOP":
        title = "Image %s - %s @%.3f A (OOP) - %s " % (
            im_id,
            topo_id,
            data,
            mode,
        )
    elif data_key == "magIP":
        title = "Image %s - %s @%.3f A (IP) - %s " % (
            im_id,
            topo_id,
            data,
            mode,
        )

    elif data_key == "energy":
        title = "Image %s - %s @%.3f eV - %s " % (
            im_id,
            topo_id,
            data,
            mode,
        )
    else:
        title = "Image %s - %s %s " % (
            im_id,
            topo_id,
            mode,
        )
    return title


In [ ]:
def plot_recon(recon, title, perc_min = 1, perc_max = 99,  scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(recon.shape)

    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(title)

    tmp = np.real(recon) * scale_mask
    vmin, vmax = np.nanpercentile(tmp[tmp != 0], (perc_min, perc_max))
    t_im1 = ax[0].imshow(np.real(recon), vmin=vmin, vmax=vmax, cmap="gray")
    ax[0].set_title("Real")
    plt.colorbar(t_im1, ax=ax[0], aspect=50)

    tmp = np.imag(recon) * scale_mask
    vmin, vmax = np.nanpercentile(tmp[tmp != 0], (perc_min, perc_max))
    t_im2 = ax[1].imshow(np.imag(recon), vmin=vmin, vmax=vmax, cmap="gray")
    ax[1].set_title("Imag")
    plt.colorbar(t_im2, ax=ax[1], aspect=50)

In [ ]:
# Save reconstruction of sum
# Saves only real and imaginary part
recon = p_pc + n_pc

# colormap scaling
footprint = skimage.morphology.disk(5)
shrink_mask = skimage.morphology.erosion(supportmask, footprint)

# Plot
title = get_title("magOOP", im_id, topo_id, CDI=True)
plot_recon(
    recon[roi_cdi_s] , title, scale_mask=shrink_mask[roi_cdi_s]
)

# Save images
fname = join(
    folder_general,
    "Recon_ImId_%04d_RefId_%s_cdi_stitching_sum_%s.png" % (im_id, topo_id, USER),
)
print("Saving: %s" % fname)
plt.savefig(fname, bbox_inches="tight", transparent=False)

In [ ]:
# Save reconstruction of difference
# Saves only real and imaginary part
recon = (p_pc - n_pc  )
#recon = (p_pc - n_pc)  / (p_pc + n_pc)
#recon = np.log(p_pc) - np.log(n_pc)

# Plot
plot_recon(
    recon[roi_cdi_s] , title, scale_mask=shrink_mask[roi_cdi_s],perc_min=1,perc_max=99
)

# Save images
fname = join(
    folder_general,
    "Recon_ImId_%04d_RefId_%s_cdi_stitching_diff_%s.png" % (im_id, topo_id, USER),
)
print("Saving: %s" % fname)
plt.savefig(fname, bbox_inches="tight", transparent=False)

# Batch processing CDI

## Define Scan Ids

In [ ]:
# Load support mask of which sample?
sample = "FBIe14"

In [ ]:
# Define the sets for reconstructions. You can make a list or use np.arange
# im_id_set should always have ids of positive helicity holograms,
# topo_id_set those of negative helicity or a proper topo

im_id_set = np.arange(3365, 3367 + 1)
im_id_set = [3373, 3374]
topo_id_set = 3370 * np.ones(len(im_id_set), dtype=int)

# In case of single helicity reconstructions, adapt the helicity
# for contrast inversion
helicity = 1 * np.ones(len(im_id_set), dtype=int)  # [1,-1]

# Do cdi?
do_cdi = True

print("Dynamics Set:  %s" % im_id_set)
print("Reference Set: %s" % topo_id_set)
print("Helicity: %s" % helicity)

## Execute Stack Reconstruction

In [ ]:
# Ugly Automatic processing of image stacks
recons_name = []  # for gifs
for it, im_id in enumerate(im_id_set):
    # Load images
    image, _ = load_processing(im_id)

    # Get topo
    # Do you want to construct topo holo from two helicity images
    try:
        len(topo_id_set[it])
    except:
        # Usual case
        # Get also topo & dark id from list of sets
        topo_id = topo_id_set[it]

        # Load data
        print(f"Loading imageId: %04d, topoId: %04d" % (im_id, topo_id))
        topo, _ = load_processing(topo_id)

        # Process images
        worker_dict = worker(image, topo)

        # Save topo hologram
        save_topo_holo(worker_dict["sum_c"], im_id, topo_id)
    else:
        print("Using Topo from sum of two helicity holograms")
        pos_id = topo_id_set[it][0]
        neg_id = topo_id_set[it][1]
        topo_id = topo_id_set[it]

        try:
            topo = load_topo_holo(pos_id, neg_id) / 2
        except:
            topo = load_topo_holo(neg_id, pos_id) / 2
        topo = cci.shift_image(
            topo, -shift_c
        )  # shift out of center so you don't need to change the worker

        # Process images
        worker_dict = worker(image, topo)

    # Save FTH reco
    # Magnetic field value
    data_key = "magnett_read"
    values = np.mean(np.array(load_data(im_id, data_key)) * 1000)
    values = [np.round(values, 2)]

    # Cryostat temperature if saved in nxs file
    try:
        data_key = "cryob"
        values.append(np.mean(np.array(load_collection(im_id, data_key))))
        title = "Image %d - %s @%.2f mT, %0.1f K" % (
            im_id,
            topo_id,
            values[0],
            values[1],
        )
    except:
        title = "Image %d - %s @%.2f mT" % (im_id, topo_id, values[0])

    # Reconstruct
    recon = fth.reconstruct(
        fth.propagate(
            worker_dict["holo"], prop_dist * 1e-6, experimental_setup=experimental_setup
        )
        * np.exp(1j * phase)
    )

    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(title)

    vmin, vmax = np.percentile(
        np.real(recon[roi[0] : roi[1], roi[2] : roi[3]]), (1, 99)
    )
    t_im1 = ax[0].imshow(
        np.real(recon[roi[0] : roi[1], roi[2] : roi[3]]),
        vmin=vmin,
        vmax=vmax,
        cmap="gray",
    )
    ax[0].set_title("Real")
    plt.colorbar(t_im1, ax=ax[0], aspect=50)

    vmin, vmax = np.percentile(
        np.imag(recon[roi[0] : roi[1], roi[2] : roi[3]]), (1, 99)
    )
    t_im2 = ax[1].imshow(
        np.imag(recon[roi[0] : roi[1], roi[2] : roi[3]]),
        vmin=vmin,
        vmax=vmax,
        cmap="gray",
    )
    ax[1].set_title("Imag")
    plt.colorbar(t_im2, ax=ax[1], aspect=50)

    # Save images
    fname = join(
        folder_general,
        "Recon_ImId_%04d_RefId_%s_fth_diff_stack_%s.png" % (im_id, topo_id, USER),
    )
    print("Saving: %s" % fname)
    plt.savefig(fname, bbox_inches="tight", transparent=False)

    ################ CDI ###############
    if do_cdi is True:
        # Create pos and neg helicity set
        pos = (worker_dict["sum_c"] + worker_dict["diff_c"]) / 2
        neg = (worker_dict["sum_c"] - worker_dict["diff_c"]) / 2

        # Create beamstop automatically
        mask_im, mask_topo, mask_pixel, mask_pixel_smooth = create_auto_beamstop(
            pos, neg, mask_draw, use_bs, bs_param
        )

        fig, ax = cimshow(mask_pixel.astype(int))
        ax.set_title("Verify that this looks like an acceptable beamstop")

        # Get coordinates and create supportmask
        support_coordinates = get_supportmask_coordinates(sample)
        supportmask = create_supportmask(support_coordinates, pos.shape)

        # Do phase retrieval
        (
            retrieved_p,
            retrieved_n,
            retrieved_p_pc,
            retrieved_n_pc,
            bsmask_p,
            bsmask_n,
            gamma_p,
            gamma_n,
        ) = phase_retrieval(
            pos, neg, mask_pixel, supportmask, Startimage=None, Startgamma=None
        )

        # Get Recos partial coherence
        # Positiv partial coherence
        p_pc = fth.reconstructCDI(
            fth.propagate(
                retrieved_p_pc * mask_bs_cdi,
                prop_dist_cdi * 1e-6,
                experimental_setup=experimental_setup,
            )
        )
        # Negative partial coherence
        n_pc = fth.reconstructCDI(
            fth.propagate(
                retrieved_n_pc * mask_bs_cdi,
                prop_dist_cdi * 1e-6,
                experimental_setup=experimental_setup,
            )
        )

        ##### Calc reco and optimze contrast
        recon = helicity[it] * (p_pc - n_pc)
        _, phase_cdi = optimize_phase_contrast(
            recon,
            supportmask,
            method="contrast",
            prefered_color="white",
        )
        # phase_cdi = 0
        recon = recon * np.exp(1j * phase_cdi)
        print("Phase is:", np.round(phase_cdi, 2))
        ########

        # Plot
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        fig.suptitle(title)

        vmin, vmax = np.percentile(np.real(recon[roi_cdi]), (0.5, 99.5))
        t_im1 = ax[0].imshow(
            np.real(recon[roi_cdi]),
            vmin=vmin,
            vmax=vmax,
            cmap="gray",
        )
        ax[0].set_title("Real")
        plt.colorbar(t_im1, ax=ax[0], aspect=50)

        vmin, vmax = np.percentile(np.imag(recon[roi_cdi]), (0.5, 99.5))
        t_im2 = ax[1].imshow(np.imag(recon[roi_cdi]), vmin=vmin, vmax=vmax, cmap="gray")
        ax[1].set_title("Imag")
        plt.colorbar(t_im2, ax=ax[1], aspect=50)

        # Save images
        fname = join(
            folder_general,
            "Recon_ImId_%04d_RefId_%s_cdi_stack_%s.png" % (im_id, topo_id, USER),
        )

        print("Saving: %s" % fname)
        plt.savefig(fname, bbox_inches="tight", transparent=False)
        recons_name.append(fname)

        # Save files as h5
        save_cdi_h5()
    else:
        print("Phase retrieval disabled!")

    print(" ")
print("CDI stack processing finished")

In [ ]:
plt.close("all")

# Gifs

In [ ]:
im_id_set = [
    1268,
    1269,
    1272,
    1273,
    1276,
    1277,
    1280,
    1281,
    1284,
    1285,
    1288,
    1289,
    1292,
    1293,
    1296,
    1297,
    1300,
    1301,
    1303,
    1306,
    1308,
    1313,
    1314,
    1315,
    1316,
    1317,
]
topo_id_set = [
    1267,
    1270,
    1271,
    1274,
    1275,
    1278,
    1279,
    1282,
    1283,
    1286,
    1287,
    1290,
    1291,
    1294,
    1295,
    1298,
    1299,
    1302,
    [1299, 1300],
    1307,
    [1306, 1307],
    1312,
    [1312, 1313],
    [1312, 1313],
    [1312, 1313],
    [1312, 1313],
]

recons_name = []
for i, im_id in enumerate(im_id_set):
    fname = join(
        folder_general,
        "Recon_ImId_%04d_RefId_%s_%s_cdi_diff_stack.png"
        % (im_id, topo_id_set[i], USER),
    )
    recons_name.append(fname)

# Create gif of last scan
var = [imageio.imread(file) for file in recons_name]
fname = f"ImId_%04d_%04d_%s.gif" % (im_id_set[0], im_id_set[-1], USER)
gif_path = path.join(folder_general, fname)
print("Saving gif:%s" % gif_path)
imageio.mimsave(gif_path, var, fps=2)
print("Done!")